# Gurobi: placement v gridu
**Recept pro praktický test KO · souřadnice, sousedství, konflikty a párování**

Nejdřív obecné postupy, potom dvě samostatně vysvětlené řešené úlohy. V textových blocích jsou šablony; skutečné kódové buňky můžeš spouštět postupně shora dolů. Řešení na konci používají pomocné funkce definované v první části.

### Co dělat pokaždé
1. Převeď vstup na **`(řádek, sloupec)` od nuly**.
2. Vytvoř binární **`x[r, c]` = je zde umístěný objekt**.
3. Zakaž nepovolená políčka pomocí **`x[r, c] == 0`**.
4. Zapiš konflikty, počty a případné přiřazení objektů.
5. Maximalizuj počet, nebo pouze najdi přípustné řešení podle zadání.
6. Zkontroluj stav, přečti `.X` a převeď souřadnice zpět.

Všude používáme **`import gurobipy as g`**, **`model`**, **`g.quicksum`** a **`for` + `addConstr`**. V těchto modelech obvykle nepotřebuješ Big-M: většina podmínek se zapíše přímo nad binárními proměnnými.


**Spuštění:** potřebuješ Python s balíčkem `gurobipy` a dostupnou licencí Gurobi. Pak spusť buňky shora dolů (Run All). Výsledky ukázek jsou již uložené i pro samotné čtení.

**Jednotný zápis:** `addVars` (s **s**) vytváří více proměnných; `addConstr` (bez **s**) přidává jedno omezení. Více omezení přidáváme cyklem `for`, takže `addConstrs` zde nepotřebujeme.


## 1. Souřadnice: jedno pravidlo v celém programu
**`H` = počet řádků, `W` = počet sloupců.** Řádek je vždy první index. Tak zvládneš i obdélníkovou mřížku.

| Co máš | Co použiješ uvnitř programu |
|---|---|
| Souřadnici `(x, y)` od 1 | `(r, c) = (y - 1, x - 1)` |
| Souřadnici `(řádek, sloupec)` od 1 | Odečti 1 od obou; neprohazuj je. |
| Šachové pole `b2` | `r = 1`, `c = 1` |
| Gurobi proměnnou na políčku | `x[r, c]` |
| Obyčejný seznam seznamů | `grid[r][c]` |

```python
# Vstupni dvojice (x, y), indexovane od 1:
positions = {(yy - 1, xx - 1) for xx, yy in input_positions}

# Zpet na (x, y), indexovane od 1:
output_positions = [(c + 1, r + 1) for r, c in selected]
```

Jména `xx, yy` zde označují vstupní souřadnice, nikoli Gurobi proměnné. **Neodvozuj pořadí souřadnic jen z názvu x/y: ověř ho v zadání nebo vizualizaci.**

U šachovnice můžeme řádek 0 kreslit dole, u běžné tabulky nahoře. To mění jen obrázek, ne model.


## 2. Sousedé = seznam posunů + kontrola hranic
Posun `(dr, dc)` se přičítá k aktuálnímu `(r, c)`. Rozlišuj tři okolí:

| Okolí | Použití |
|---|---|
| 4 sousedé | Vodorovný/svislý dotyk. |
| 8 sousedů | Vodorovný, svislý i diagonální dotyk. |
| 8 tahů jezdce | Posun o 1 a 2 v různých osách. |

Následující helper vrací **jen existující políčka**. Procházíš tedy bezpečně rohy i kraje a podmínku hranic píšeš pouze jednou.


In [1]:
import gurobipy as g

DIR4 = [(-1, 0), (1, 0), (0, -1), (0, 1)]
DIR8 = [(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1)
        if (dr, dc) != (0, 0)]
KNIGHT = [(-2, -1), (-2, 1), (-1, -2), (-1, 2),
          (1, -2), (1, 2), (2, -1), (2, 1)]

def neighbors(r, c, H, W, moves):
    result = []
    for dr, dc in moves:
        rr, cc = r + dr, c + dc
        if 0 <= rr < H and 0 <= cc < W:
            result.append((rr, cc))
    return result

# Roh, stred a obdelnik: odhalime chyby jeste bez solveru.
assert set(neighbors(0, 0, 3, 5, DIR4)) == {(1, 0), (0, 1)}
assert len(neighbors(1, 1, 3, 5, DIR8)) == 8
assert set(neighbors(0, 0, 3, 5, KNIGHT)) == {(1, 2), (2, 1)}
print("Sousedstvi a hranice: OK")


Sousedstvi a hranice: OK


## 3. Proměnné, překážky, řádky a sloupce
**Nejjednodušší výchozí styl: vytvoř proměnnou pro každé políčko.** Proměnné na překážkách zafixuj na nulu. Vyhneš se chybějícím klíčům a součty zůstanou jednoduché.

```python
model = g.Model()
x = model.addVars(H, W, vtype=g.GRB.BINARY, name="x")

# blocked je mnozina zakazanych dvojic (r, c).
for r, c in blocked:
    model.addConstr(x[r, c] == 0)

# row_count[r] udava presny pocet v radku r.
for r in range(H):
    model.addConstr(g.quicksum(x[r, c] for c in range(W)) == row_count[r])

# col_count[c] udava presny pocet ve sloupci c.
for c in range(W):
    model.addConstr(g.quicksum(x[r, c] for r in range(H)) == col_count[c])
```

**Vnější cyklus určuje jedno omezení; vnitřní cyklus sčítá jeho členy.** Pro řádek se mění sloupec. Pro sloupec se mění řádek. Je-li požadováno „nejvýše“, použij `<=` místo `==`.

U přesných počtů musí platit `sum(row_count) == sum(col_count)`. Je to nutná kontrola, nikoli záruka řešitelnosti.

**Pozor na chybný trik:** `if (r, c) in blocked: continue` samo políčko nezakáže. Pokud jsi už pro něj vytvořil proměnnou, musíš ji opravdu zafixovat na 0.


## 4. Zákaz konfliktu: dvě políčka nesmějí být obsazena současně
Pro každou konfliktní dvojici přidej:
```python
model.addConstr(x[r, c] + x[rr, cc] <= 1)
```
Není potřeba `if x[r, c] == 1`. Řešič obě hodnoty teprve hledá.

V helperu níže používáme **celý symetrický seznam tahů** a porovnání dvojic `(r, c) < (rr, cc)`. Python je porovnává nejdřív podle řádku a potom podle sloupce. Díky tomu vytvoříme každou neorientovanou dvojici jen jednou.

Pokud bys použil pouze polovinu tahů a zároveň toto porovnání, některé konflikty můžeš omylem vynechat. **Drž se jednoho způsobu: všechny tahy + porovnání dvojic.**


In [2]:
def add_conflicts(model, x, H, W, moves):
    # moves musi obsahovat oba smery kazdeho posunu.
    for r in range(H):
        for c in range(W):
            for rr, cc in neighbors(r, c, H, W, moves):
                if (r, c) < (rr, cc):
                    model.addConstr(x[r, c] + x[rr, cc] <= 1)

# Pouziti po vytvoreni modelu a promennych (vyber jedno):
# add_conflicts(model, x, H, W, DIR4)
# add_conflicts(model, x, H, W, DIR8)
# add_conflicts(model, x, H, W, KNIGHT)
print("Helper add_conflicts je pripraven.")


Helper add_conflicts je pripraven.


### Dva užitečné triky a jedna past
**1. Okolí určuje zadání.** Blízkost nemusí být konflikt: někdy je požadovaná. Pro různé podmínky můžeš potřebovat různé seznamy posunů.

**2. Pro zákaz všech 8 dotyků lze použít bloky 2 × 2:** v každém takovém bloku smí být nejvýše jeden objekt.
```python
for r in range(H - 1):
    for c in range(W - 1):
        model.addConstr(x[r, c] + x[r + 1, c]
                        + x[r, c + 1] + x[r + 1, c + 1] <= 1)
```
Pro `H >= 2` a `W >= 2` má tato alternativa stejnou množinu binárních řešení jako zákaz všech sousedních dvojic a silnější lokální LP omezení. Pro mřížku s jediným řádkem či sloupcem nestačí: nevznikne žádný blok. Univerzální a doporučený základ zůstává `add_conflicts`.

**Past: součet celého okolí 3 × 3 <= 1 je příliš silný.** Zakáže i dva objekty, které leží na opačných stranách okolí a navzájem se nedotýkají. Nezaměň jej se zákazem každé konfliktní dvojice.


## 5. Sousedství není přiřazení
Obecně rozlišuj tři výroky. `S` je množina existujících sousedních polí:

| Výrok | Zápis |
|---|---|
| V okolí je alespoň jeden objekt | `model.addConstr(g.quicksum(x[p] for p in S) >= 1)` |
| V okolí je právě jeden objekt | Totéž s `== 1`. |
| Tomuto bodu je přiřazen právě jeden objekt | Potřebuješ proměnné přiřazení, pokud samotná přítomnost nestačí. |

`x[p]` je totéž jako `x[r, c]`, pokud `p` je dvojice `(r, c)`.

Pokud se mají objekty **párovat 1:1**, obyčejné „každý má nějakého souseda“ nezaručuje správné přiřazení: několik objektů se může spoléhat na tentýž protějšek. Použij binární proměnné pro povolené vazby a jejich součty. Konkrétní úplný příklad je až v části B.

### Kdy vytvářet proměnné jen na povolených polích?
Pro základní obsazení doporučujeme plnou mřížku. U vazeb je naopak přehledné mít jen možné kombinace:
```python
# E je seznam povolenych trojic (id_bodu, radek, sloupec).
link = model.addVars(E, vtype=g.GRB.BINARY, name="link")
# Pristup: link[t, r, c]. Neexistujici trojice vyvola KeyError.
```
Podobně `model.addVars(cells, ...)` vytvoří proměnné pro seznam dvojic polí. Pak ale musíš filtrovat každý přístup pomocí `if p in cells`. Pro malý grid je plná mřížka zpravidla jednodušší.


## 6. Cíl, výsledek a ladění
```python
# Pokud chces umistit co nejvice objektu:
model.setObjective(
    g.quicksum(x[r, c] for r in range(H) for c in range(W)),
    g.GRB.MAXIMIZE
)

# Pokud jen hledas splneni presne zadanych podminek:
# model.setObjective(0, g.GRB.MINIMIZE)

model.optimize()
```
**Vyber jeden cíl.** Jsou-li počty v řádcích pevné, celkový počet je již určený; maximalizovat jej nic nepřidá.

Po nalezení řešení přečti vybraná políčka přes `.X > 0.5`, nikoli přes `if x[r, c]`. Při nesplnitelnosti `.X` nečti. Níže helper vrací `True` pro optimum, `False` pro prokázanou nesplnitelnost a pro ostatní stavy chybu; časový limit automaticky nepovažuje za nesplnitelnost.


In [3]:
def solve_status(model):
    model.optimize()
    if model.Status == g.GRB.INF_OR_UNBD:
        model.Params.DualReductions = 0
        model.optimize()
    if model.Status == g.GRB.OPTIMAL:
        return True
    if model.Status == g.GRB.INFEASIBLE:
        return False
    raise RuntimeError(f"Neocekavany stav solveru: {model.Status}")

def selected_cells(x, H, W):
    # Volat jen tehdy, kdy existuje reseni.
    return [(r, c) for r in range(H) for c in range(W)
            if x[r, c].X > 0.5]


### Checklist pro grid
- Zkontroluj malý **obdélník**, roh a okraj; čtverec skryje prohozené rozměry.
- `grid[-1][c]` v Pythonu znamená poslední řádek! Hranice kontroluj **před přístupem**. Gurobi klíč s indexem -1 obvykle vůbec neexistuje.
- Objekty ze vstupu jsou **konstanty**, umisťované objekty jsou **proměnné**. `if` nad vstupem je v pořádku.
- Nepoužívej `[[0] * W] * H`: řádky sdílí stejný seznam. Použij `[[0 for c in range(W)] for r in range(H)]`.
- Ohraničení pomocí pomocného rámu `H+2 × W+2` také funguje, ale rámové **rozhodovací proměnné musí být fixované na 0**, nesmí vstupovat do cíle ani výstupu. Zde proto používáme kontrolu hranic bez rámu.
- Překážka někdy zakazuje jen své pole, jindy i okolí nebo celý řádek. Nehádej to podle obrázku, čti pravidlo.
- Vypiš nebo vykresli řešení. Přepočítej součty a konflikty **z vybraných souřadnic**, nezávisle na modelu.
- Při podezřelém výsledku pomůže `model.write("debug.lp")`. Pro `INFEASIBLE` lze zavolat `model.computeIIS()` a `model.write("conflict.ilp")`.

**Co si zapamatovat:** dvě smyčky přes grid → seznam posunů → kontrola hranic → jedno omezení pro každou příslušnou dvojici.


## Pomocný obrázek výsledku
Následující buňku pouze spusť. Vytvoří barevnou HTML tabulku pro notebook; neovlivňuje optimalizaci. Souřadnice v obrázku jsou vždy interní `(r, c)` od nuly. U tabulek je řádek 0 nahoře, u šachovnice jej zobrazíme dole.


In [4]:
from IPython.display import display, HTML
from html import escape

def show_grid(H, W, fixed, chosen, fixed_label="T", chosen_label="X",
              row_count=None, col_count=None, bottom_zero=False, title=""):
    fixed, chosen = set(fixed), set(chosen)
    style = "border:1px solid #bac8d3;width:32px;height:32px;text-align:center;color:#172b4d"
    parts = ["<div><b>" + escape(title) + "</b></div>",
             '<table style="border-collapse:collapse;font-family:monospace">']
    parts.append("<tr><th>r / c</th>" + "".join(f"<th>{c}</th>" for c in range(W)) + "<th></th></tr>")
    rows = range(H - 1, -1, -1) if bottom_zero else range(H)
    for r in rows:
        parts.append(f"<tr><th>{r}</th>")
        for c in range(W):
            p = (r, c)
            label = fixed_label if p in fixed else chosen_label if p in chosen else ""
            color = "#bce3cc" if p in fixed else "#ffd78b" if p in chosen else "#f4f7fa"
            parts.append(f'<td style="{style};background:{color}">{escape(label)}</td>')
        parts.append("<th>" + (str(row_count[r]) if row_count is not None else "") + "</th></tr>")
    if col_count is not None:
        parts.append("<tr><th>Pocet</th>" + "".join(f"<th>{v}</th>" for v in col_count) + "<th></th></tr>")
    parts.append("</table>")
    display(HTML("".join(parts)))


# A. Vyřešený příklad: jezdci a pevné věže
**Příloha `kings.pdf` se uvnitř jmenuje „Knights Placement“: umisťujeme jezdce, nikoli krále.**

Na šachovnici 8 × 8 jsou předem dané věže. Umísti maximální počet jezdců tak, aby se žádní dva neohrožovali a žádný nebyl ohrožen věží. Podle tohoto konkrétního zadání věž zakazuje **celý svůj řádek a sloupec včetně vlastního pole**; neřešíme blokování jinými figurami.

Naše ukázka má věže **b2 a d5**. Očekávaný maximální počet je **20**. Bez věží je optimum **32**. Jsou-li všechna pole zakázaná, správný výsledek je **0**, nikoli nesplnitelnost.

Model: `x[r,c]` je přítomnost jezdce; zakázané řádky/sloupce fixujeme na 0, konflikty vznikají z `KNIGHT`. Funkce používá obecné helpery z první části.


In [5]:
def solve_knights(rooks, H=8, W=8):
    # rooks uz obsahuje (radek, sloupec) od 0.
    model = g.Model("knights")
    model.Params.OutputFlag = 0
    x = model.addVars(H, W, vtype=g.GRB.BINARY, name="x")

    for rr, cc in rooks:
        for r in range(H):
            model.addConstr(x[r, cc] == 0)
        for c in range(W):
            model.addConstr(x[rr, c] == 0)

    add_conflicts(model, x, H, W, KNIGHT)
    model.setObjective(
        g.quicksum(x[r, c] for r in range(H) for c in range(W)),
        g.GRB.MAXIMIZE
    )
    try:
        if not solve_status(model):
            return None
        return selected_cells(x, H, W)
    finally:
        model.dispose()

def from_chess(s):
    return int(s[1:]) - 1, ord(s[0]) - ord("a")

def to_chess(r, c):
    return chr(ord("a") + c) + str(r + 1)

rooks = [from_chess(s) for s in ["b2", "d5"]]
knights = solve_knights(rooks)
assert len(knights) == 20
print("Pocet jezdcu:", len(knights))
print("Policka:", [to_chess(r, c) for r, c in knights])
show_grid(8, 8, rooks, knights, fixed_label="V", chosen_label="J",
          bottom_zero=True, title="V = pevna vez, J = nalezeny jezdec")


Restricted license - for non-production use only - expires 2027-11-29
Pocet jezdcu: 20
Policka: ['a1', 'c1', 'e1', 'f1', 'g1', 'h1', 'a3', 'a4', 'e4', 'f4', 'g4', 'h4', 'a6', 'c6', 'h7', 'a8', 'c8', 'e8', 'g8', 'h8']


r / c,0,1,2,3,4,5,6,7,
7,J,,J,,J,,J,J,
6,,,,,,,,J,
5,J,,J,,,,,,
4,,,,V,,,,,
3,J,,,,J,J,J,J,
2,J,,,,,,,,
1,,V,,,,,,,
0,J,,J,,J,J,J,J,


### Přesný vstup a výstup této úlohy
Vstup: na prvním řádku počet věží, pak každá souřadnice zvlášť, například `2`, `b2`, `d5` na třech řádcích.

```python
with open(sys.argv[1], "r") as f:
    count = int(f.readline())
    rooks = [from_chess(f.readline().strip()) for _ in range(count)]

knights = solve_knights(rooks)
if knights is None:
    raise RuntimeError("Prazdne rozmisteni ma byt vzdy pripustne.")
with open(sys.argv[2], "w") as f:
    f.write(str(len(knights)) + "\n")
    for r, c in knights:
        f.write(to_chess(r, c) + "\n")
```
Pro odevzdání vytvoř `main.py` s importy a použitými funkcemi, na první řádek dej `#!/usr/bin/env python3` a spusť `python3 main.py input.txt output.txt`. V notebooku nepoužívej `sys.argv[1]` pro vlastní vstup; argumenty tam patří kernelu.


# B. Vyřešený příklad: stany a stromy
Na mřížce jsou pevné stromy. Najdi umístění stanů podle těchto pravidel:
1. Stan stojí na volném poli **vodorovně nebo svisle vedle stromu**.
2. Každý strom je spárovaný právě s jedním stanem a každý stan právě s jedním stromem.
3. Stany se navzájem nedotýkají, ani diagonálně.
4. Počty stanů v každém řádku a sloupci jsou zadané přesně.
5. Strom může sousedit s více stany, ale **přiřazený** má jen jeden.

V dodaném notebooku jsou souřadnice stromů **`(x, y)` od 1**; potvrzuje to zápis `tree_grid[t_y][t_x]`. Převedeme je na `(r,c)` od 0. Jeho nejmenší příklad má `n=3`, i když komentář omylem říká „2x2“.

### Proč nelze napsat „u každého stromu právě jeden sousední stan“?
Na obrázku níže má strom na `(1,1)` dva sousední stany, a přesto je řešení správné: levý stan přiřadíme dolnímu stromu, pravý prostřednímu. Stanům nevadí vzdálenost dvou polí ve stejném řádku.

Stejně tak obecně nestačí „u každého stromu alespoň jeden stan“ a správný celkový počet; musíme zaručit, že se jeden stan nepoužije pro více stromů.


In [6]:
demo_trees = {(1, 1), (2, 0)}
demo_tents = {(1, 0), (1, 2)}
show_grid(3, 3, demo_trees, demo_tents, fixed_label="T", chosen_label="S",
          row_count=[0, 2, 0], col_count=[1, 0, 1],
          title="Platne reseni: T = strom, S = stan")
print("Parovani: strom (2,0) -> stan (1,0); strom (1,1) -> stan (1,2)")


Parovani: strom (2,0) -> stan (1,0); strom (1,1) -> stan (1,2)


r / c,0,1,2,
0,,,,0
1,S,T,S,2
2,T,,,0
Pocet,1,0,1,


### Model: obsazení pole + vazby
- `x[r,c] ∈ {0,1}`: na políčku stojí stan.
- `link[t,r,c] ∈ {0,1}`: strom s indexem `t` je přiřazen stanu na `(r,c)`.
- `E`: pouze povolené vazby strom → volný čtyřsoused. Jsou to známé trojice, nikoli proměnné.

**Pro každý strom:** součet jeho vazeb je 1.

**Pro každé políčko:** součet vazeb do něj je `x[r,c]`. Tím obsazené pole dostane právě jeden strom a neobsazené žádný. Pole bez možné vazby má automaticky `x[r,c]=0`.

Pak přidáme řádkové a sloupcové počty, zákaz stanů na stromech a konflikty podle `DIR8`. Počet stanů je daný; cíl může být konstanta 0.

Součty s podmínkou `if` níže filtrují **známý seznam vazeb**. Netestují hodnoty rozhodovacích proměnných.


In [7]:
def solve_tents(H, W, row_count, col_count, trees):
    # trees: seznam (radek, sloupec) od 0; kazdy strom jen jednou.
    trees = list(trees)
    assert len(row_count) == H and len(col_count) == W
    assert len(trees) == len(set(trees))
    assert all(0 <= r < H and 0 <= c < W for r, c in trees)
    tree_set = set(trees)

    # Nejdriv pripravime vsechny legalni vazby podle vstupnich dat.
    E = []
    for t, (r, c) in enumerate(trees):
        for rr, cc in neighbors(r, c, H, W, DIR4):
            if (rr, cc) not in tree_set:
                E.append((t, rr, cc))

    model = g.Model("tents")
    model.Params.OutputFlag = 0
    x = model.addVars(H, W, vtype=g.GRB.BINARY, name="x")
    link = model.addVars(E, vtype=g.GRB.BINARY, name="link")

    for r, c in trees:
        model.addConstr(x[r, c] == 0)
    for r in range(H):
        model.addConstr(g.quicksum(x[r, c] for c in range(W)) == row_count[r])
    for c in range(W):
        model.addConstr(g.quicksum(x[r, c] for r in range(H)) == col_count[c])
    add_conflicts(model, x, H, W, DIR8)

    # Kazdy strom si vybere prave jednu povolenou vazbu.
    for t in range(len(trees)):
        model.addConstr(g.quicksum(link[key] for key in E if key[0] == t) == 1)

    # Na vybrane pole vede prave jedna vazba; na ostatni zadna.
    for r in range(H):
        for c in range(W):
            model.addConstr(g.quicksum(link[t, rr, cc] for t, rr, cc in E
                                      if rr == r and cc == c) == x[r, c])

    model.setObjective(0, g.GRB.MINIMIZE)
    try:
        if not solve_status(model):
            return None
        tents = selected_cells(x, H, W)
        pairs = [(trees[t], (r, c)) for t, r, c in E if link[t, r, c].X > 0.5]
        return tents, pairs
    finally:
        model.dispose()


### Malý příklad z přílohy (3 × 3)
Stromy jsou na vstupních souřadnicích `(1,1)` a `(3,2)`. Počty řádků jsou `[1,1,0]`, počty sloupců `[1,0,1]`. První souřadnice vstupu je sloupec. Výsledek převedeme zpět do stejného formátu.


In [8]:
n = 3
row_count, col_count = [1, 1, 0], [1, 0, 1]
trees_xy = [(1, 1), (3, 2)]
trees_rc = [(yy - 1, xx - 1) for xx, yy in trees_xy]

result = solve_tents(n, n, row_count, col_count, trees_rc)
assert result is not None
tents, pairs = result
print("Stany (x,y) od 1:", [(c + 1, r + 1) for r, c in tents])
print("Parovani (r,c) od 0:", pairs)
show_grid(n, n, trees_rc, tents, fixed_label="T", chosen_label="S",
          row_count=row_count, col_count=col_count, title="Stany: 3 x 3")


Stany (x,y) od 1: [(3, 1), (1, 2)]
Parovani (r,c) od 0: [((0, 0), (1, 0)), ((1, 2), (0, 2))]


r / c,0,1,2,
0,T,,S,1
1,S,,T,1
2,,,,0
Pocet,1,0,1,


### Střední příklad z přílohy (8 × 8)
Stejný model a stejné převody. Měníme pouze data. Nepouštěj v původním notebooku postupně několik datových buněk s očekáváním, že se řeší všechny: poslední buňka přepíše `n`, `r`, `c` i `trees`. Tady jsou jednotlivé příklady vyřešené hned po zadání dat.


In [9]:
n8 = 8
rows8 = [3, 1, 1, 2, 0, 2, 0, 3]
cols8 = [2, 1, 2, 2, 1, 1, 2, 1]
trees8_xy = [(2, 1), (5, 1), (6, 1), (1, 2), (3, 3), (3, 4),
             (6, 4), (4, 5), (6, 5), (8, 7), (2, 8), (4, 8)]
trees8_rc = [(yy - 1, xx - 1) for xx, yy in trees8_xy]
result8 = solve_tents(n8, n8, rows8, cols8, trees8_rc)
assert result8 is not None
tents8, pairs8 = result8
print("Pocet stanu:", len(tents8))
print("Stany (x,y) od 1:", [(c + 1, r + 1) for r, c in tents8])
show_grid(n8, n8, trees8_rc, tents8, fixed_label="T", chosen_label="S",
          row_count=rows8, col_count=cols8, title="Stany: 8 x 8")


Pocet stanu: 12
Stany (x,y) od 1: [(1, 1), (3, 1), (7, 1), (5, 2), (2, 3), (4, 4), (7, 4), (4, 6), (6, 6), (1, 8), (3, 8), (8, 8)]


r / c,0,1,2,3,4,5,6,7,
0,S,T,S,,T,T,S,,3
1,T,,,,S,,,,1
2,,S,T,,,,,,1
3,,,T,S,,T,S,,2
4,,,,T,,T,,,0
5,,,,S,,S,,,2
6,,,,,,,,T,0
7,S,T,S,T,,,,S,3
Pocet,2,1,2,2,1,1,2,1,


## 7. Kontroly nezávislé na modelu
Tuto buňku můžeš jen spustit. Ověřuje souřadnice, součty, konflikty i to, že každá vazba opravdu spojuje sousední strom a stan a že se žádný nepoužije dvakrát. K testům patří i případ, kdy jeden strom správně sousedí se dvěma stany.


In [10]:
def check_knights(H, W, rooks, selected):
    assert selected is not None
    occupied = set(selected)
    assert len(occupied) == len(selected)
    for r, c in occupied:
        assert 0 <= r < H and 0 <= c < W
        assert all(r != rr and c != cc for rr, cc in rooks)
        for rr, cc in occupied:
            assert (abs(r - rr), abs(c - cc)) not in [(1, 2), (2, 1)]

def check_tents(H, W, rows, cols, trees, result):
    assert result is not None, "Ocekavame reseni."
    tents, pairs = result
    assert len(tents) == len(set(tents)) == len(trees)
    assert all(0 <= r < H and 0 <= c < W for r, c in tents)
    assert not (set(tents) & set(trees))
    assert [sum(r == rr for rr, cc in tents) for r in range(H)] == list(rows)
    assert [sum(c == cc for rr, cc in tents) for c in range(W)] == list(cols)
    for i, (r, c) in enumerate(tents):
        for rr, cc in tents[i + 1:]:
            assert max(abs(r - rr), abs(c - cc)) > 1
    assert len(pairs) == len(trees)
    assert len({tree for tree, tent in pairs}) == len(trees)
    assert len({tent for tree, tent in pairs}) == len(tents)
    assert {tree for tree, tent in pairs} == set(trees)
    assert {tent for tree, tent in pairs} == set(tents)
    assert all(abs(a[0]-b[0]) + abs(a[1]-b[1]) == 1 for a, b in pairs)

# Tri priklady z PDF: 32, 20 a 0 jezdcu.
for rook_names, expected in [([], 32), (["b2", "d5"], 20),
    (["a1", "b2", "c3", "d4", "e5", "f6", "g7", "h2"], 0)]:
    fixed = [from_chess(s) for s in rook_names]
    chosen = solve_knights(fixed)
    check_knights(8, 8, fixed, chosen)
    assert len(chosen) == expected
print("Jezdci: vsechny 3 priklady OK")

check_tents(3, 3, [1,1,0], [1,0,1], trees_rc, result)
check_tents(8, 8, rows8, cols8, trees8_rc, result8)
special = solve_tents(3, 3, [0,2,0], [1,0,1], [(1,1),(2,0)])
check_tents(3, 3, [0,2,0], [1,0,1], [(1,1),(2,0)], special)
assert set(special[0]) == {(1,0),(1,2)}

# Obdelnik a nesplnitelny pripad: kontrola hranic a stavu.
rectangle = solve_tents(2, 4, [1,0], [0,1,0,0], [(0,0)])
check_tents(2, 4, [1,0], [0,1,0,0], [(0,0)], rectangle)
assert solve_tents(2, 2, [0,0], [0,0], [(0,0)]) is None
print("Stany: priklady, parovani, obdelnik a nesplnitelnost OK")


Jezdci: vsechny 3 priklady OK
Stany: priklady, parovani, obdelnik a nesplnitelnost OK


## 8. Zkus to bez opisování
**Krátká vlastní úloha:** na gridu **3 × 4** umísti co nejvíce značek. Značky se nesmí dotýkat ani diagonálně. Pole `(1,1)` je zakázané. Souřadnice jsou `(řádek, sloupec)` od 0.

Napiš model v následující buňce. Výsledek ulož do seznamu `answer_practice`. Potřebuješ: binární tabulku, zákaz jednoho pole, konflikty `DIR8`, maximalizaci součtu, vyřešení a čtení `.X`. Správný počet je **4**; rozmístění může být různé.


In [11]:
# DOPLN: cely model nebo pouziti helperu vyse.
# Po vyreseni uloz souradnice do answer_practice.
answer_practice = None

# Kontrola se spusti, az doplnis reseni.
if answer_practice is None:
    print("Dobrovolne cviceni: dopln vlastni model.")
else:
    assert len(answer_practice) == len(set(answer_practice)) == 4
    assert (1, 1) not in answer_practice
    assert all(0 <= r < 3 and 0 <= c < 4 for r, c in answer_practice)
    for i, (r, c) in enumerate(answer_practice):
        for rr, cc in answer_practice[i+1:]:
            assert max(abs(r-rr), abs(c-cc)) > 1
    print("Vlastni model: OK")


Dobrovolne cviceni: dopln vlastni model.


### Podklady a co si vzít k testu
Podklady: dodaný `[Student]_Lab_04_ILP2_(Tents_in_the_Forest).ipynb` a `kings.pdf` (uvnitř *Knights Placement*). Všechny potřebné podmínky řešených příkladů jsou zopakované přímo výše. Obrázky používají T/S a V/J místo ikon, aby byly souřadnice snadno čitelné.

Oficiální API: [Model, addVars, addConstr](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html), [quicksum](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/func_global.html), [stavy optimalizace](https://docs.gurobi.com/projects/optimizer/en/current/reference/numericcodes/statuscodes.html).

**Minimum k zapamatování:** `x[r,c]`, `neighbors`, `add_conflicts`, součty po řádcích/sloupcích a rozlišení **obsazení vs. přiřazení**. Zbytek je přizpůsobení vstupu, pravidel a výstupu.
